# Assignment - Visualization

Make sure to **read the instructions** completely. Failing to follow the instructions will have a negative impact on your grade!

In this assignment, we will explore ways to visualize the data, starting with simple plots for early exploration and build towards more interactive methods. As we want to go beyond static plots possible with for example matplotlib or seaborn, we will use the [Bokeh](https://www.bokeh.org/) library. This means the exercises **must** be implemented with Bokeh to receive points.

This assignment includes four parts (Assignmets V.1 to V.4) which provide different amounts of points. V.1 to V.3 are independent assignments, but V.4 builds on the previous assignments.
 
**Before you submit**, make sure that you are not accidentaly using any global variables. Restart the kernel (wiping all global variables) and run the code from top to bottom by clicking "Kernel" => "Restart & Run all" in the menu bar at the top.

### API documentation

note: you can find the used versions by checking the `environment.yml` file or `running conda list -n dsait4000vis` from the command line

* [Bokeh](https://docs.bokeh.org/) (also has examples)
* [Pandas](https://pandas.pydata.org/docs/)
* [NumPy](https://numpy.org/doc/stable/)
* [SciPy](https://docs.scipy.org/doc/scipy/)


### Rules
For this assignment the following rules apply:

**General**
 * The assignment should be completed in the **groups** you have already assembled for the previous assignment. Work together on all parts and do not distribute the work. Everyone is responsible for everything. That also means if there is fraud in a part it will be considered fraud by the whole group and voids the complete assignment, even if you distributed the parts of the assignment.
 * Any kind of intergroup discussion will be considered fraud and both the parties will be punished.
 * All code must be written within group. All external help, with the exception of Python/library documentation and the lecture slides, will be considered fraud.
 * Do not add any libarires to those already included notebook (i.e., `numpy`, `scipy`, `pandas`, and `bokeh`)
 * Add functionality from bokeh, like ```from bokeh.io import output_notebook```, on top of the block where you need it.
 * **Do not rename the notebook file**

**AI Use**
 * The use of AI for this assignment is permitted. If you use AI, please explain what AI was used for and how you made sure you used it responsibly in the section at the end of this file. 
 * Every group member should be able to explain all facets of the entire assignment. 

**Grading**
 * Exercise may (partially) be graded automatically, using additional hidden tests. These tests will check your solution for different (unseen) inputs.
 * If these tests pass without error (warnings are allowed) then you receive full points.
 * Do not make any assumptions on the input data (such as number or type of columns) unless specified otherwise. Doing so may result in the tests failing and thus 0 points.
 * Your grade is calculated as the sum of the achieved points and will be rounded to the closest 0.1 point.
 
**Submissions**
 * Submit your code to Brightspace
 * For this assignment you must *only* submit the notebook (`*.ipynb`) file filled in with your solution.
 * Refer to Brightspace for the exact deadline.
 * We will consider submissions that are handed in up to 3 hours after the deadline but with a penalty as follows:
 * For every completed 20 minutes past the deadline the grade will be reduced by 0.5 (e.g., with the deadline at 12:00, a submission at 12:19 will get the full grade, a submission at 12:20 will receive a 0.5 grade penalty).

In [1]:
# We do the basic setup here.
# Do not (!) change this cell! All changes will be reset during grading!

# numpy and pandas
import numpy as np
import pandas as pd

# We import a range of bokeh functionality that will likely be needed already
# if you need more, import it at the top of the corresponding cell
from bokeh.plotting import figure, gridplot, show
from bokeh.layouts import column
from bokeh.transform import factor_cmap, linear_cmap
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter, BasicTicker, PrintfTickFormatter, CustomJS, Select, axes
from bokeh.models.annotations import Label
from bokeh.io import output_notebook
output_notebook() # activate Bokeh output to Jupyter notebook

# You can use this to calculate the density plot in Assignment V.3
from scipy.stats import gaussian_kde

# We load the adult data  (https://archive.ics.uci.edu/dataset/2/adult) as in previous assignments (note, we keep the headers, to use them in the pandas dataframe)
# Note: we do not change ? values to nan, but rather keep them as ? so they will be shown in the visualizations
data = pd.read_csv("datasets/Visualization/adult_all.csv")

data.head()

Loading BokehJS ...

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Assignment V.1 (5 points)

Getting an overview of the data distribution

In this first assignment, we want to use visualization to get a quick overview of the distribution of the different attributes/columns. This can often be helpful to make sure that we use models that match the distribution downstream or to quickly identify for example whether the data is distributed over all classes equally if the attribute is intended to be used for training a model.

A good way to quickly plot the distribution is a simple bar chart for each attribute. However, this will only allow us to look at one attribute at a time. In class we have discussed small multiples. We can use this concept to plot all attributes in a grid of plots. 

For this assignment do the following:
* Create a grid of bar charts, with one chart per column in the dataset. The attribute values shall be on the x-axis and the count of each value on the y-axis. Look at pandas `value_counts` function to get the latter.
  Hint: The `figure()` function works slightly different for numerical and categorical data.
* Make sure that the charts and axes are well labeled (for example like in the screenshot below). It will be hard to properly label the x-axis with many items (e.g., native-country). Do this as well as you can, you will likely not find a perfect solution for all attributes.
* Add a tooltip to show the values on hover (see the education chart, top-right, in the image below).

The result should look similar to the below.

[<img src="assets/barcharts.png" alt="Small Multiples showing the distributions of all columns in the dataset." width="500"/>](assets/barcharts.png)

In [2]:
# YOUR CODE HERE
def make_hist(data, col, title=None, label_map=None, axis_labels=True):
    s = data[col]
    if label_map:
        s = s.astype(str).str.strip().replace(label_map) 
        
    counts = s.value_counts() # count each value in that col

    if pd.api.types.is_numeric_dtype(s): # numerical
        counts = counts.sort_index() # sort in ascending order
        x = counts.index.to_list() # index's to list
        width = (max(x) - min(x)) / len(x) * 0.9 if len(x) > 1 else 1 
        p = figure(width=300, height=250, title=title or col)
        p.xaxis.formatter = NumeralTickFormatter(format="0,0")
    else: # categorical
        if label_map:
            counts = counts.sort_index()
        x = counts.index.astype(str).str.strip().to_list()
        width = 0.5
        p = figure(width=300, height=250, title=title or col, x_range=x)
    source = ColumnDataSource(dict(x=x, top=counts.values)) # for tooltip
    p.vbar(x="x", top="top", width=width, source=source) # actual bar chart

    if axis_labels:
        p.yaxis.axis_label = "count"
    p.xaxis.major_label_orientation = 1.0 # x axis label rotation
    p.xaxis.major_label_text_font_size = "7pt"
    p.yaxis.formatter = NumeralTickFormatter(format="0")
    p.add_tools(HoverTool(tooltips=[(col, "@x"), ("count", "@top")]))
    return p

    
data = data.assign(salary=data["salary"].str.strip().str.rstrip("."))

others = [make_hist(data, col) for col in data.columns if col != "salary"]
income = make_hist(
    data, "salary",
    title="Income",
    label_map={"<=50K": "<50K"},
    axis_labels=False,
)

show(gridplot(others + [income], ncols=4))

#### A bar chart like this doesn't serve every attribute equally well

Look closely at the grid you just built. For some attributes, a few attribute values accounts for the overwhelming majority of the data, resulting in a few very long bars. The remaining attribute values  Such a bar chart does not allow us to see the distribution of the data in a meaningful way. 

* Identify the numerical attributes where this happens. For this task, you can select attributes by name. 
* For just those attributes, rebuild the chart as a **binned histogram**. Do not make any changes to the other bar charts. You may need to try a few different bin sizes to find a suitable one. **Justify your choice for the bin sizes in 50 words or less** in the response field below. 
* Choose an appropriate scale for the Y axis that shows the data distribution properly. **Justify your choice for the Y axis scale in 50 words or less** in the response field below.  

In [3]:
# YOUR CODE HERE
cols = ["capital-gain", "capital-loss", "hours-per-week"]
bins = {"capital-gain" : 40, "capital-loss": 40, "hours-per-week": 40}

def make_binned_hist(data, col, nbins):
    counts, edges = np.histogram(data[col], bins=nbins) # split into bins
    keep = counts > 0
    counts = counts[keep] # only non-empty bins 
    left = edges[:-1][keep] # every boundary except last
    right = edges[1:][keep] # every boundary excpet first
    
    p = figure(width=450, height=300, title=col, y_axis_type="log") # log y-axis
    p.xaxis.formatter = NumeralTickFormatter(format="0,0") 
    source = ColumnDataSource(dict(top=counts, left=left, right=right)) # tooltip
    p.quad(top="top", bottom=0.5, left="left", right="right", source=source) # draw bar
    p.yaxis.axis_label = "count (log scaled)"
    p.add_tools(HoverTool(tooltips=[("Range", "@left{0,0} to @right{0,0}"), ("count", "@top")]))
    
    return p


plots = [make_binned_hist(data, c, bins[c]) for c in cols] 
show(gridplot(plots, ncols=2))

#### Response: Justification for binning strategy and Y axis scale 

YOUR ANSWER HERE

Many bins give clear separation because individual bin range becomes narrower, revealing more detail in the data distribution. Lower bin amount leads to merged information, hiding structure. After testing, it became apparent that nbin of 20 is less informative than one of 40. More was not considered due to noisiness.

For y-axis, a logarithmic scale was chosen because for every column, a count is present which represents majority of the data. Therefore, if not for log-scale, the bar with majority values would flatten all the other bars on a linear scale. Log fixes that and makes small counts visible.


### Assignment V.2 (5 points)

Visualizing possible correlations and some interactivity

In this assignment, we want to visualize the correlation between attributes. To keep this simple we only use the numerical variables here. You can use pandas `corr()` [function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html) directly on the dataframe using the `numeric_only = True` argument. The output will contain only the results from using numerical columns. It should be noted though, that there are also ways to calculate dependencies or correlation between catagorical (e.g., [Chi-Square_test](https://en.wikipedia.org/wiki/Chi-square_test), [Cramér's V](https://en.wikipedia.org/wiki/Cramér%27s_V)) or categorical and numerical (e.g., [One-way ANOVA test](https://en.wikipedia.org/wiki/F-test#One-way_ANOVA_example)) values.

The output will be a matrix of pairwise values. It is common to visualize such data in a heatmap (images below).

* Calculate the correlation of the numerical columns using pandas `corr()`, using Pearson correlation (this is the default of the function).
* Create a heatmap showing the values. The data format output by `corr()` might not be very useful. Look for example at [pandas `stack()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html) for possible ways to transforma the data into a typical dataframe.
* A heatmap uses x and y position for the attributes and color for the value, choose a fitting colormap and **justify it below in 50 words or less**. (Note: the color chosen in the example below might or might not be good. Do **not** use the example to justify your choice.)
* Make sure the chart is legible by adding axes, legends, a color bar, or similar. It is up to you to decide what is needed.

We now want to extend our chart to be able to switch between the Pearson and Spearman correlation interactively using a dropdown menu directly in the chart.
* Add a `Select` dropdown with the two options.
* Calculate both correlation values and add a plot for each to your figure.
* Use `js_on_change` on the added Select to add code to switch between the two plots.

[<img src="assets/heatmap.png" alt="Heatmap showing correlations" width="500"/>](assets/heatmap.png)
[<img src="assets/heatmap_select.png" alt="Heatmap showing correlations with open selection widget" width="500"/>](assets/heatmap_select.png)


In [4]:
# We calculate both matrices
correlation_matrices = {
    "pearson": data.corr(numeric_only = True, method="pearson"),
    "spearman": data.corr(numeric_only = True, method="spearman")
}

# this is the default correlation method
correlation_method = "spearman"


# YOUR CODE HERE
from bokeh.palettes import RdBu11
from bokeh.models import ColorBar

def make_heatmap(corr, method):
    cols = list(corr.columns)
    df = corr.stack().reset_index()
    df.columns = ["x", "y", "value"]
    source = ColumnDataSource(df)

    p = figure(width=550, height=500, title=method + " correlation", x_range=cols, y_range=cols[::-1])
    colours = linear_cmap("value", palette=RdBu11, low=-1, high=1)
    p.rect(x="x", y="y", width=1, height=1, source=source, fill_color=colours, line_color="white")

    p.xaxis.axis_label = "attribute"
    p.yaxis.axis_label = "attribute"
    p.xaxis.major_label_orientation = 0.8
    p.add_tools(HoverTool(tooltips=[("x", "@x"), ("y", "@y"), ("correlation", "@value{0.00}")]))

    bar = ColorBar(color_mapper=colours["transform"], title="correlation")
    p.add_layout(bar, "right")
    return p

pearson_plot = make_heatmap(correlation_matrices["pearson"], "pearson")
spearman_plot = make_heatmap(correlation_matrices["spearman"], "spearman")

pearson_plot.visible = correlation_method == "pearson"
spearman_plot.visible = correlation_method == "spearman"

select = Select(title="Correlation method", value=correlation_method, options=["pearson", "spearman"])
select.js_on_change("value", CustomJS(args=dict(p1=pearson_plot, p2=spearman_plot), code="""
    p1.visible = cb_obj.value == "pearson";
    p2.visible = cb_obj.value == "spearman";
"""))

show(column(select, pearson_plot, spearman_plot))


#### Response: Justification for colormap

Correlation ranges from −1 to +1 with 0 meaning no relation, so we used a red–blue colormap with white in the middle: blue for negative, red for positive, and darker colours for stronger correlation. Fixing the range to −1..1 keeps 0 white in both plots.

### Assignment V.3 (5 points)

In this assignment, we will visualize the relationship between two numerical columns and one categorical column from the dataset. We already extract the names of the first two numerical columns below; the easiest is to just use those, but instead of fixing the categorical column we now want it to be **user-selectable**, similar to Assignment V.2. (Remember that we might test this with a different dataset, so do not pick columns by name.)

The goal here is to visualize the complete data of the selected columns. We will proceed in two steps again.

In the first step, create a scatterplot to visualize the selected columns.
* A scatterplot is a common representation, bokeh provides it using `scatter`.
* Use the x and y axes for the numerical values.
* Add a `Select` dropdown listing the categorical columns in the dataset, and use it to choose which categorical column is encoded in the plot (e.g. through color). Use `js_on_change` to update the plot when a different column is selected, similar to Assignment V.2.
* Choose a proper encoding for the categorical column and **justify** it in the field below the code **in 50 words or less**. (Note: this encoding was omitted in the example figure below.)
* Make sure the chart is legible by adjusting mark sizes, adding axes, legends, a color bar, or similar, etc. It is up to you to decide what is needed.

Even with putting in a lot of effort, you will notice that there will be a lot of overplotting, even with 'just' ~50k points in the dataset. In the example below, you see iso-density lines added to show how the points distribute. We have already added the `kde` function to [estimate the density.](https://en.wikipedia.org/wiki/Multivariate_kernel_density_estimation) Use this to calculate the point density. This is taken from an [example in the Bokeh documenation](https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html). Look at the [example](https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html) and adjust it to your needs.
* Add the density information to your plot. (This only needs to depend on the two numerical columns, not on the selected categorical column.)

[<img src="assets/scatterplot.png" alt="Scatterplot with density indicator" width="700"/>](assets/scatterplot.png)


In [5]:
# kde calculation taken from https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html
# look at the example on how it is used. You can use the plot code also for inspiration, but make sure to adjust it for proper results
# note, N is the kernel size, larger values will make this lengthy to calculate but give more smooth results.
# When you use it consider using small values for N during testing.
def kde(x, y, N):

    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()

    X, Y = np.mgrid[xmin:xmax:N*1j, ymin:ymax:N*1j]
    positions = np.vstack([X.ravel(), Y.ravel()])
    values = np.vstack([x, y])
    kernel = gaussian_kde(values)
    Z = np.reshape(kernel(positions).T, X.shape)

    return X, Y, Z

# we extract the names of the first two numerical columns
num_cols = []
num_cols.append(data.select_dtypes(include="number").columns[0])
num_cols.append(data.select_dtypes(include="number").columns[1])
# and all categorical columns, to populate the dropdown.
cat_cols = list(data.select_dtypes(include=["object", "category"]).columns)
# You can just use the above columns to set up your visualization. Feel free to change these for testing.
# For the submission, you can safely assume that at least 2 numerical and 1 categorical columns are available in the dataset.

# YOUR CODE HERE
from bokeh.palettes import Category10
from bokeh.layouts import row
from bokeh.models import Div, Slider, CDSView, CustomJSFilter, BasicTickFormatter, Spinner

x_col, y_col = num_cols

def get_colours(col):
    values = data[col].astype(object).fillna("Unknown").astype(str).replace("?", "Unknown")
    counts = values.value_counts()
    colour_map = {}
    if len(counts) > 10:
        top = list(counts.index[:9])
        values = values.where(values.isin(top), "Other")
        colour_map["Other"] = "#000000"
    else:
        top = list(counts.index)
    for i, v in enumerate(top):
        colour_map[v] = Category10[10][i]
    legend = "<b>" + col + "</b><br>" + "<br>".join(
        '<span style="color:' + c + '">&#11044;</span> ' + v for v, c in colour_map.items())
    return values.tolist(), values.map(colour_map).tolist(), legend

source = ColumnDataSource(data={"x": data[x_col], "y": data[y_col]})
legends = {}
for col in cat_cols:
    labels, colours, legends[col] = get_colours(col)
    source.data[col + "_label"] = labels
    source.data[col + "_colour"] = colours
source.data["colour"] = source.data[cat_cols[0] + "_colour"]    
source.data["rank"] = np.random.permutation(len(data))          


slider = Slider(start=1, end=len(data), value=len(data), step=1, title="Points shown")
only_first_n = CustomJSFilter(args=dict(slider=slider), code="""
    return Array.from(source.data["rank"]).map(r => r < slider.value);
""")
slider.js_on_change("value", CustomJS(args=dict(source=source), code="source.change.emit();"))


p = figure(width=900, height=600, title=y_col + " vs " + x_col, x_axis_label=x_col, y_axis_label=y_col)
p.scatter("x", "y", source=source, size=4, alpha=0.4, color="colour", view=CDSView(filter=only_first_n))
p.xaxis.formatter = BasicTickFormatter(use_scientific=False)
p.yaxis.formatter = BasicTickFormatter(use_scientific=False)


numbers = data[[x_col, y_col]].dropna()
X, Y, Z = kde(numbers[x_col], numbers[y_col], 100)
p.contour(X, Y, Z, levels=np.linspace(Z.min(), Z.max(), 12)[1:-1], line_color="black")


legend = Div(text=legends[cat_cols[0]])
select = Select(title="Colour by", value=cat_cols[0], options=cat_cols)
select.js_on_change("value", CustomJS(args=dict(source=source, legend=legend, legends=legends), code="""
    source.data["colour"] = source.data[cb_obj.value + "_colour"];
    source.change.emit();
    legend.text = legends[cb_obj.value];
"""))
number_box = Spinner(title="Or type a number", low=1, high=len(data), step=100, value=len(data), mode="int")
slider.js_link("value", number_box, "value")
number_box.js_link("value", slider, "value")

show(row(column(select, slider, number_box, legend), p))

#### Response: Justification for categorical data encoding

Column values have no order, so we give each its own hue to distinguish between them. We used 10 colours, not 20, since lighter and darker shades clash with the 40% opacity that shows overlapping points. Rare values go into a black “Other”. Missing and “?” values become “Unknown”.

### Assignment V.4 (10 points)

Interactive exploration

In this final assignment we want to do some true interactive exploration, harnessing the power of Bokeh by combining selection and filtering interactions with the charts you already built.
You have two options to proceed, based on your previous results. The below descriptions are minimal solutions, feel free to go beyond the suggested functionality, if you deem it useful. If you do so add a brief list of what you added in the textfield below.

#### Pick one of the the following options:

1. Use the result of **Assignment V.1** and extend it such that clicking a bar in one of the charts affects all the charts. I.e.,
   * Make it possible to select any subset of the data by clicking the corresponding bar in any of the bar/histogram charts. E.g., clicking on the *Bachelors* bar in the *Education* chart should select all datapoints where the value of Education is Bachelors.
   * Render the effect of that selection in every chart (while keeping the full dataset visible too). E.g., render a second bar corresponding to the fraction of the corresponding points within a different color on top of the original bar.
<br/><br/>
2. Combine the results of **Assignment V.2** and **Assignment V.3** into a single linked layout.
   * Show both plots next to each other in a grid.
   * Allow selection of the numerical columns in the scatterplot by clicking on a cell in the heatmap.
   * Reuse or extend the categorical-column dropdown you already built in Assignment V.3.
   * Add a way to select one or multiple values of the categorical column and update the scatterplot based on that selection. E.g., show only the datapoints where *Education* equals *Bachelors*, when Education is selected as the categorical variable of the plot. Ensure that the axis ranges do not change when the user selects values of the categorical column so that plotted points do not shift. 

For your chosen option, explain the analysis task(s) that your interactive visualization allows for. Describe it as one or more **{action, target}** pairs. Say briefly why the interaction you built (rather than a static version of the same charts) is what makes that task possible, **in 50 words or less**.

#### Some possibly useful hints:
* Check out **TapTool** for more information on enabling clicking events on plots
* Have a look at **CDSView** for one possible way to filter out a subset of data on a plot
* [Enabling click callbacks on bars](https://docs.bokeh.org/en/2.4.3/docs/user_guide/tools.html#setting-the-active-tools)
* [Getting clicked bar index in callback functions](https://discourse.bokeh.org/t/a-different-plot-in-another-figure-when-a-bar-of-a-barchart-is-clicked/8572/2)
* [Apply hover tools to plots](https://stackoverflow.com/questions/29435200/bokeh-plotting-enable-tooltips-for-only-some-glyphs)


In [6]:
# YOUR CODE HERE
from bokeh.models import TapTool, Button, Div


def build_linked_charts(data, hist_bins):
    hist_bins = {c: b for c, b in hist_bins.items()
                 if c in data.columns and pd.api.types.is_numeric_dtype(data[c])
                 and not pd.api.types.is_bool_dtype(data[c])}

    figs, sources, is_log = [], [], []
    row_codes = {}

    for j, col in enumerate(data.columns):
        s = data[col]
        tools = "reset"

        if col in hist_bins:
            values = s.to_numpy(dtype=float)
            counts, edges = np.histogram(values[~np.isnan(values)], bins=hist_bins[col])
            codes = np.digitize(values, edges[1:-1])
            codes[np.isnan(values)] = -1
            keep = counts > 0
            new_index = np.cumsum(keep) - 1
            codes = np.where(codes >= 0, new_index[np.clip(codes, 0, None)], -1)

            source = ColumnDataSource(dict(
                left=edges[:-1][keep], right=edges[1:][keep], top=counts[keep],
                sel=np.full(keep.sum(), np.nan), pct=["-"] * int(keep.sum())))
            p = figure(width=300, height=250, title=col, y_axis_type="log", tools=tools)
            base = p.quad(left="left", right="right", bottom=0.5, top="top", source=source,
                          fill_color="#9ecae1", line_color="white",
                          selection_fill_color="#3182bd", selection_line_color="black",
                          nonselection_fill_color="#9ecae1", nonselection_fill_alpha=1.0,
                          nonselection_line_color="white")
            p.quad(left="left", right="right", bottom=0.5, top="sel", source=source,
                   fill_color="#e6550d", fill_alpha=0.85, line_color=None,
                   selection_fill_color="#e6550d", selection_fill_alpha=0.85,
                   nonselection_fill_color="#e6550d", nonselection_fill_alpha=0.85)
            p.yaxis.axis_label = "count (log scaled)"
            hover = HoverTool(renderers=[base], tooltips=[
                (col, "@left{0,0} to @right{0,0}"), ("count", "@top{0,0}"),
                ("selected", "@sel{0,0}"), ("share selected", "@pct")])

        else:
            if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
                x = np.sort(s.dropna().unique())
                codes = pd.Categorical(s, categories=x).codes
                counts = s.value_counts().reindex(x).to_numpy()
                width = (x.max() - x.min()) / len(x) * 0.9 if len(x) > 1 else 1
                p = figure(width=300, height=250, title=col, tools=tools)
            else:
                vals = s.astype(str).where(s.notna())
                vc = vals.value_counts()
                x = vc.index.to_list()
                codes = pd.Categorical(vals, categories=x).codes
                counts = vc.to_numpy()
                width = 0.8
                p = figure(width=300, height=250, title=col, x_range=x, tools=tools)

            source = ColumnDataSource(dict(
                x=np.asarray(x).tolist(), top=counts, sel=np.zeros(len(x)), pct=["-"] * len(x)))
            base = p.vbar(x="x", top="top", width=width, source=source,
                          fill_color="#9ecae1", line_color="white",
                          selection_fill_color="#3182bd", selection_line_color="black",
                          nonselection_fill_color="#9ecae1", nonselection_fill_alpha=1.0,
                          nonselection_line_color="white")
            p.vbar(x="x", top="sel", width=width, source=source,
                   fill_color="#e6550d", fill_alpha=0.85, line_color=None,
                   selection_fill_color="#e6550d", selection_fill_alpha=0.85,
                   nonselection_fill_color="#e6550d", nonselection_fill_alpha=0.85)
            p.yaxis.axis_label = "count"
            p.yaxis.formatter = NumeralTickFormatter(format="0,0")
            hover = HoverTool(renderers=[base], tooltips=[
                (col, "@x"), ("count", "@top{0,0}"),
                ("selected", "@sel{0,0}"), ("share selected", "@pct")])

        tap = TapTool(renderers=[base])
        p.add_tools(hover, tap)
        p.xaxis.axis_label = col
        p.xaxis.major_label_orientation = 1.0
        p.xaxis.major_label_text_font_size = "7pt"

        row_codes["c" + str(j)] = np.asarray(codes, dtype=np.int32)
        figs.append(p)
        sources.append(source)
        is_log.append(col in hist_bins)

    rows = ColumnDataSource(row_codes)
    info = Div(text="<b>No selection.</b> Click a bar to select those rows "
                    "(shift+click for several bars). Selections in different charts are combined (AND).")
    reset_button = Button(label="Clear selection", button_type="primary", width=150)

    update = CustomJS(args=dict(sources=list(sources), rows=rows, is_log=list(is_log), info=info), code="""
        const k = sources.length;
        const n = rows.data["c0"].length;
        const active = [];
        for (let j = 0; j < k; j++) {
            if (sources[j].selected.indices.length > 0) active.push(j);
        }
        const mask = new Uint8Array(n).fill(1);
        for (const j of active) {
            const chosen = new Set(sources[j].selected.indices);
            const codes = rows.data["c" + j];
            for (let i = 0; i < n; i++) {
                if (mask[i] && !chosen.has(codes[i])) mask[i] = 0;
            }
        }
        let total = 0;
        for (let i = 0; i < n; i++) total += mask[i];

        for (let j = 0; j < k; j++) {
            const src = sources[j];
            const top = src.data["top"];
            const codes = rows.data["c" + j];
            const counts = new Array(top.length).fill(0);
            if (active.length > 0) {
                for (let i = 0; i < n; i++) {
                    if (mask[i] && codes[i] >= 0) counts[codes[i]] += 1;
                }
            }
            const sel = [], pct = [];
            for (let b = 0; b < top.length; b++) {
                sel.push((is_log[j] && counts[b] === 0) ? NaN : counts[b]);
                pct.push(active.length > 0 ? (100 * counts[b] / top[b]).toFixed(1) + "%" : "-");
            }
            src.data["sel"] = sel;
            src.data["pct"] = pct;
            src.change.emit();
        }
        if (active.length === 0) {
            info.text = "<b>No selection.</b> Click a bar to select those rows " +
                        "(shift+click for several bars). Selections in different charts are combined (AND).";
        } else {
            info.text = "<b>" + total.toLocaleString() + "</b> of " + n.toLocaleString() +
                        " rows selected (" + (100 * total / n).toFixed(1) + "%). " +
                        "Orange = selected part of each bar.";
        }
    """)
    for src in sources:
        src.selected.js_on_change("indices", update)

    reset_button.js_on_click(CustomJS(args=dict(sources=list(sources)), code="""
        for (const src of sources) src.selected.indices = [];
    """))

    grid = gridplot(figs, ncols=4, toolbar_location=None)
    return column(info, reset_button, grid)


show(build_linked_charts(data.assign(salary=data["salary"].str.rstrip(".")),{"fnlwgt": 40, "capital-gain": 40, "capital-loss": 40, "hours-per-week": 40}))

#### Response: Task(s) your visualization supports

Task 1: {Compare, distributions}: how every attribute is distributed in a chosen subset vs. the whole population. {Discover, dependencies}: For example, select >50K and see how education, occupation and hours shift. Since static charts only show marginal distributions, clicking recomputes every chart for any combined subset, revealing conditional distributions.

Added beyond the minimum:

1. Shift+click selects several bars in one chart (OR operation)
2. Selections across charts combine with AND
3. Status line showing number and percentage of rows selected
4. “Clear selection” button
5. Hover shows each bar’s selected count and share selected
6. fnlwgt shown as a 40-bin histogram (unlike V.1): almost every value is unique, so its bars were invisible and unclickable

#### AI Use

YOUR RESPONSE HERE

We used AI for data understanding and also to reframe/summarize our response
